# ZC702 On-Device Check

Connects to the ZC702 board over SSH and runs real on-device inference - not a description of what the board does, an actual live connection to it. Requires the board to be powered on and reachable (Ethernet + Windows ICS, or update `BOARD_IP` below to match however it's connected right now).

Background: the board's default `tflite_runtime` (the only wheel ever published for this platform) crashes with `Illegal instruction` on this exact CPU (Cortex-A9, `vfpv3` but no `vfpv4`/FMA - the wheel assumes `vfpv4`). Fixed by building `tflite_runtime` from source, natively on the board. Full story: `fpga/README.md`.

In [1]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

PLINK = r"C:\Program Files\PuTTY\plink.exe"
PSCP = r"C:\Program Files\PuTTY\pscp.exe"
BOARD_IP = "192.168.137.136"  # update if the board's DHCP lease has changed - it has no fixed MAC/EEPROM
BOARD_USER = "xilinx"
BOARD_PASS = "xilinx"
HOSTKEY = "SHA256:Ar2+h5ZV4JYSIOh21xf6yzHMqZFq9loxKvv4X9yGJ5M"


def ssh(cmd, timeout=60):
    result = subprocess.run(
        [PLINK, "-ssh", "-batch", "-pw", BOARD_PASS, "-hostkey", HOSTKEY,
         f"{BOARD_USER}@{BOARD_IP}", cmd],
        capture_output=True, text=True, timeout=timeout,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:\n", result.stderr)
    return result

## 1. Confirm the board is up and the working runtime is installed

In [2]:
result = ssh("uptime; cat /proc/cpuinfo | grep Features | head -1")
ssh("python3 -c \"from tflite_runtime import interpreter; print('tflite_runtime import OK:', interpreter.__file__)\"")

 00:49:16 up  2:53,  1 user,  load average: 1.00, 1.03, 1.06
Features	: half thumb fastmult vfp edsp neon vfpv3 tls vfpd32 



tflite_runtime import OK: /home/xilinx/.local/lib/python3.6/site-packages/tflite_runtime/interpreter.py



CompletedProcess(args=['C:\\Program Files\\PuTTY\\plink.exe', '-ssh', '-batch', '-pw', 'xilinx', '-hostkey', 'SHA256:Ar2+h5ZV4JYSIOh21xf6yzHMqZFq9loxKvv4X9yGJ5M', 'xilinx@192.168.137.136', 'python3 -c "from tflite_runtime import interpreter; print(\'tflite_runtime import OK:\', interpreter.__file__)"'], returncode=0, stdout='tflite_runtime import OK: /home/xilinx/.local/lib/python3.6/site-packages/tflite_runtime/interpreter.py\n', stderr='')

## 2. Run a real inference on-device (webcam)

Captures a live frame from the USB webcam right now, runs it through the fine-tuned model, and prints the prediction. `~9s` per inference is expected - XNNPACK is disabled on this ARM32 build for safety, so it's running on the plain reference kernels.

In [3]:
ssh("cd /home/xilinx && python3 webcam_infer.py", timeout=30)

FRAME_SHAPE=(480, 640, 3)
INFERENCE_TIME_S=9.14
PREDICTED_CLASS=texting_left CONFIDENCE=0.898
END_TO_END_OK



CompletedProcess(args=['C:\\Program Files\\PuTTY\\plink.exe', '-ssh', '-batch', '-pw', 'xilinx', '-hostkey', 'SHA256:Ar2+h5ZV4JYSIOh21xf6yzHMqZFq9loxKvv4X9yGJ5M', 'xilinx@192.168.137.136', 'cd /home/xilinx && python3 webcam_infer.py'], returncode=0, stdout='FRAME_SHAPE=(480, 640, 3)\nINFERENCE_TIME_S=9.14\nPREDICTED_CLASS=texting_left CONFIDENCE=0.898\nEND_TO_END_OK\n', stderr='')

## 3. On-device accuracy, on real labelled images

A quick 5-image spot check (~45s at ~9s/image) using images already on the board. The full 300-image run this project's headline numbers are based on takes ~45 minutes at this speed, so it's not reproduced live here - see `fpga/README.md` for the full on-device accuracy numbers.

In [4]:
quick_check_script = '''import sys, csv
sys.path.insert(0, '/home/xilinx/tflite_rebuild/tflite_runtime_pkg')
from tflite_runtime import interpreter as tflite
import numpy as np, cv2

CLASS_NAMES = [f"c{i}" for i in range(10)]
interp = tflite.Interpreter(model_path='/home/xilinx/mobilenetv2_crossview_finetuned_int8.tflite')
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

rows = list(csv.DictReader(open('/home/xilinx/sample_images_100/manifest.csv')))[:5]
correct = 0
for row in rows:
    bgr = cv2.imread(f"/home/xilinx/sample_images_100/{row['filename']}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_LINEAR)
    x = resized.astype(np.float32)[None, ...]
    interp.set_tensor(inp['index'], x)
    interp.invoke()
    y = interp.get_tensor(out['index'])[0]
    pred = CLASS_NAMES[int(np.argmax(y))]
    ok = pred == row['classname']
    correct += int(ok)
    print(f"{row['filename']:30s} true={row['classname']} pred={pred} conf={y.max():.3f} {'OK' if ok else 'WRONG'}")
print(f"\\n{correct}/5 correct on this quick sample")
'''

local_script = Path.cwd() / "_quick_check_temp.py"
local_script.write_text(quick_check_script)

subprocess.run([PSCP, "-batch", "-pw", BOARD_PASS, "-hostkey", HOSTKEY, str(local_script),
                f"{BOARD_USER}@{BOARD_IP}:/home/xilinx/_quick_check_temp.py"],
               capture_output=True, text=True, timeout=30)
local_script.unlink()

result = ssh("cd /home/xilinx && python3 _quick_check_temp.py", timeout=90)

c0_0_0_img_54393.jpg           true=c0 pred=c0 conf=0.887 OK
c0_0_1_img_27079.jpg           true=c0 pred=c0 conf=0.996 OK
c0_0_2_img_94863.jpg           true=c0 pred=c0 conf=0.582 OK
c0_0_3_img_59964.jpg           true=c0 pred=c0 conf=0.973 OK
c0_0_4_img_8868.jpg            true=c0 pred=c0 conf=0.957 OK

5/5 correct on this quick sample



## Headline numbers (from the full on-device runs - see `fpga/README.md` for the summary)

| | Baseline (frozen-backbone) | Fine-tuned |
|---|---|---|
| On-device, 300-image sample | 54.7% | **77.3%** |
| Board-vs-reference-runtime agreement (matched preprocessing) | - | **100/100** |
| Inference speed | ~9.15s/image | ~9.15s/image |

The 100/100 agreement check is the important one for correctness: it proves the from-source `tflite_runtime` build on this board produces numerically identical predictions to the official reference runtime, not just "doesn't crash."